# Data analysis

In [ ]:
from pathlib import Path
import pandas as pd

# Resolve the project root from wherever the kernel started (works whether the
# cwd is notebooks_my/ or the repo root), then point at the generated-data dir.
ROOT = Path.cwd()
while not (ROOT / "data" / "plasmidscope_primary").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PP = ROOT / "data" / "plasmidscope_primary"

# ---------------------------------------------------------------------------------------------------
# What each generated dataset is, and when we made it (all live in data/plasmidscope_primary/).
# Grouped by the pipeline stage that produced it; dates are when the file was written.
#
# STAGE 1 — PlasmidScope source + working set  (2026-07-09; characterize_plasmidscope_all.py,
#           assemble_working_set.py)
#   all_metadata               PlasmidScope's deduplicated ALL table — 852,600 source plasmids + metadata (our entry point)
#   working_set                the 208,248 complete, non-lab-made plasmids we actually analyse (funnel output)
#   complete_provenance        per working-set plasmid: which source DBs it was consolidated from (provenance sets)
#   discarded_labmade          audit log of the 112 plasmids dropped as cloning/expression/shuttle vectors
#   analysis_table             analysis-ready derived view of the working set
#
# STAGE 2 — Environment & geography  (2026-07-09/10; recover_env_local.py, fetch_sra_env.py,
#           reconcile_environment.py)
#   env_local                  raw IMG/PR GOLD ecosystem labels per plasmid
#   sra_env_raw                raw mMGE SRA sample environment labels
#   fetch_insdc_targets        INSDC/BioSample accessions queued for environment lookup
#   environment_reconciled     harmonised habitat taxonomy (hab_top / hab_sub / is_clinical) — the LOCKED taxonomy
#
# STAGE 3 — PLSDB curated enrichment  (2026-07-09; enrich_plsdb_metadata.py)
#   plsdb_enrichment           PLSDB curated layer for the 45,743 PLSDB-matched plasmids
#                              (Inc typing, geography, AMRFinderPlus genes, ecosystem/disease tags)
#
# STAGE 4 — PlasAnn functional annotation  (2026-07-14; harvest_plasann.py, aggregate_plasann_genes.py)
#   plasann_features           per-plasmid annotation summary (CDS, replicons, oriV/oriT, functional-category counts)
#   plasann_gene_freq          gene-name frequency across the set
#   plasann_replicon_freq      PlasAnn replicon frequencies
#   plasann_replicon_family_freq   PlasAnn replicon-family frequencies
#   plasann_category_totals    totals per functional category
#   plasann_feature_types      counts per feature type
#
# STAGE 5 — Replicon/Inc + MOB-suite typing  (2026-07-16/20; plasmidfinder_run.sbatch,
#           mobtyper_array.sbatch, harvest_typing.py)
#   typing_authoritative       authoritative PlasmidFinder Inc + mob_typer calls per plasmid (pf_*, mob_*)
#   mob_full                   full mob_typer output (all fields) per plasmid
#
# STAGE 6 — Master table + CARD/RGI AMR  (2026-07-23; build_metadata_master.py, harvest_card.py,
#           aggregate_card.py)
#   plasmid_metadata_master    THE core table: 208,248 plasmids x 67 cols, every layer above folded in
#   card_hits                  one row per ORF-ARG hit, all cut-offs (Perfect / Strict / Loose)
#   card_per_plasmid           one row per AMR-carrier plasmid (Perfect + Strict only)
#
# STAGE 7 — Phase-3 One Health AMR analysis result tables  (2026-07-27; analyze_amr_onehealth_mobility.py)
#   onehealth_prevalence_by_compartment / _by_mobility   AMR prevalence + Wilson 95% CIs
#   onehealth_compartment_x_mobility                     joint prevalence (Simpson's-paradox check)
#   onehealth_pairwise_vs_natural                        Fisher vs natural-environment baseline, BH-corrected
#   onehealth_logit_odds_ratios                          logistic-regression odds ratios
#   onehealth_drug_class_by_compartment                  drug-class spectrum per compartment
#   onehealth_arg_jaccard / onehealth_arg_richness       cross-compartment ARG-repertoire sharing + richness
#   onehealth_multireplicon_amr                          multireplicon (>=2 Inc) vs AMR per compartment
#   onehealth_dereplication_sensitivity                  1-per-MOB-cluster clonal-redundancy robustness
#
# STAGE 8 — Round-1 peer-review reassessment result tables  (2026-07-27; reassess_review_findings.py)
#   reassess_host_coverage / reassess_host_adjustment    host-taxon confounder coverage + OR shift
#   reassess_vif / reassess_cluster_robust               collinearity (VIF) + cluster-robust SEs
#   reassess_assignment_bias                             AMR in compartment-assigned vs unassigned plasmids
#   reassess_typing_coverage                             PlasmidFinder coverage per compartment
#   reassess_clinical_basis                              what the is_clinical flag rests on
#   reassess_headline_contrast                           adjusted predicted P(AMR+) at matched plasmid size
# ---------------------------------------------------------------------------------------------------

# Load every table above into `data`, keyed by filename stem.
data = {}
for f in sorted(PP.glob("*.tsv")) + sorted(PP.glob("*.tsv.gz")):
    key = f.name[:-7] if f.name.endswith(".tsv.gz") else f.name[:-4]
    data[key] = pd.read_csv(f, sep="\t", low_memory=False)

# Convenient aliases for the tables you'll reach for most.
master           = data["plasmid_metadata_master"]  # STAGE 6 core table — 208,248 plasmids x 67 cols
plasmidscope     = data["all_metadata"]             # STAGE 1 PlasmidScope ALL (deduplicated source metadata)
working_set      = data["working_set"]              # STAGE 1 the 208,248-plasmid analysis set
provenance       = data["complete_provenance"]      # STAGE 1 source-DB membership per plasmid
environment      = data["environment_reconciled"]   # STAGE 2 locked habitat taxonomy
plsdb            = data["plsdb_enrichment"]          # STAGE 3 PLSDB curated layer (45,743 plasmids)
plasann          = data["plasann_features"]          # STAGE 4 per-plasmid PlasAnn annotation summary
typing           = data["typing_authoritative"]      # STAGE 5 PlasmidFinder + mob_typer calls
card_hits        = data["card_hits"]                 # STAGE 6 one row per ORF-ARG hit (all cut-offs)
card_per_plasmid = data["card_per_plasmid"]          # STAGE 6 one row per AMR-carrier plasmid

print(f"loaded {len(data)} tables from {PP}")
for k, v in data.items():
    print(f"  {k:42s} {v.shape[0]:>8,} x {v.shape[1]:>2}")

In [ ]:
import matplotlib.pyplot as plt

# master["sources"] is the set of source DBs each plasmid came from -> 102 distinct combinations.
vc = master["sources"].value_counts()
top = vc.head(20)   # the tail beyond ~20 is tiny; plot the 20 most common

fig, ax = plt.subplots(figsize=(8, 6))
top[::-1].plot.barh(ax=ax, color="#2a78d6")          # largest at top
for i, v in enumerate(top[::-1]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=8)
ax.set_xlabel("plasmids")
ax.set_ylabel("source database(s)")
ax.set_title(f"master['sources'].value_counts() — top 20 of {vc.size} combinations")
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

In [ ]:
# cols_of_interest is a LIST of column names (each a separate quoted string), assigned with `=`.
cols_of_interest = ['env_status', 'hab_top', 'hab_sub', 'hab_channel', 'is_clinical']
master[cols_of_interest]   # returns those 5 columns as a DataFrame

In [ ]:
plas_check = plasmidscope[(plasmidscope["Completeness"] == "complete")]
plas_check["Host"].value_counts() 

In [ ]:
# Join PlasmidScope's observed Host onto master via the provenance rep->full key bridge:
#   master.plasmid_id = the representative id (first source);
#   PlasmidScope keys Host by the full comma-joined consolidated id.
# Two chained .map()s = the two hops (rep -> full -> Host). Host is sparse (~46% of the working
# set) and Enterobacterales-skewed, so treat it as partial-coverage metadata, not a clean column.
NULL_HOST = {"", "-", "unclassified", "uncultured bacterium", "synthetic construct"}

rep2full = dict(zip(provenance.plasmid_id.str.split(",").str[0], provenance.plasmid_id))
host_map = dict(zip(plasmidscope.Plasmid_ID, plasmidscope.Host))

host = master.plasmid_id.map(rep2full).map(host_map)             # rep -> full -> Host
host = host.where(~host.fillna("").str.strip().isin(NULL_HOST))  # blank out null-ish labels
master["host"] = host
master["host_genus"] = host.str.split().str[0]                  # first whitespace token = genus

n = master["host"].notna().sum()
print(f"host populated for {n:,} / {len(master):,} plasmids ({100*n/len(master):.1f}%); "
      f"{master.host_genus.nunique():,} distinct genera")
master["host_genus"].value_counts().head(10)

In [ ]:
# ------------------------------------------------------------------------------------------------
# STANDING ANALYSIS BASE — use `real` (not `master`) for every analysis from here on.
# Drops the two non-habitat artifact buckets (simulated communities + lab cultures), matching the
# paper's locked analysis set. Filter on hab_top, NOT geo_source: geo_source=='img_gold_simulated'
# is only 64,649 and would leave 9 simulated plasmids in.
# ------------------------------------------------------------------------------------------------
ARTIFACTS = ["Simulated-artifact", "Lab-artifact"]
real = master[~master["hab_top"].isin(ARTIFACTS)].copy()

print(f"analysis base `real`: {len(real):,} / {len(master):,} plasmids "
      f"(dropped {len(master)-len(real):,}: "
      f"{(master.hab_top=='Simulated-artifact').sum():,} simulated + "
      f"{(master.hab_top=='Lab-artifact').sum():,} lab-artifact)")

In [ ]:
# hab_top's ONLY job was to draw the analysis set above (the simulated/lab-artifact exclusion). It is
# NOT an analysis variable — One Health compartments are built from hab_sub + is_clinical, never hab_top.
# Drop it from `real` so it can't be reached downstream. (master keeps the column; only `real` loses it.)
# errors="ignore" makes this safe to re-run.
real = real.drop(columns="hab_top", errors="ignore")
print(f"`real`: {real.shape[0]:,} x {real.shape[1]} cols — hab_top dropped; "
      f"habitat fields kept: {[c for c in ['hab_sub','hab_channel','is_clinical'] if c in real.columns]}")

In [ ]:
real

In [ ]:
# Host-species distribution (from the join above): plot the top 20 species.
hs = real["host"].value_counts()          # full Host string = species level (null-ish already removed)
top = hs.head(20)
fig, ax = plt.subplots(figsize=(8, 6))
top[::-1].plot.barh(ax=ax, color="#1baf7a")          # largest at top
for i, v in enumerate(top[::-1]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=8)
ax.set_xlabel("plasmids")
ax.set_ylabel("host species")
ax.set_title(f"Host species — top 20 of {hs.size:,} species "
             f"({master['host'].notna().sum():,} plasmids with a host)")
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

In [ ]:
# Host by country — for the plasmids that carry BOTH a host and a country label.
# Row-normalised host-genus composition of the 12 countries with the most hosted plasmids.
hc = real[real["host_genus"].notna() & real["geo_country"].notna()]
print(f"{len(hc):,} of {len(real):,} `real` plasmids have both a host genus and a country")

top_countries = hc["geo_country"].value_counts().head(12).index
top_genera    = hc["host_genus"].value_counts().head(8).index

sub = hc[hc["geo_country"].isin(top_countries)].copy()
sub["genus"] = sub["host_genus"].where(sub["host_genus"].isin(top_genera), "Other")

ct   = pd.crosstab(sub["geo_country"], sub["genus"])[list(top_genera) + ["Other"]]
ct   = ct.loc[top_countries[::-1]]                 # biggest country last -> ends up at top in barh
frac = ct.div(ct.sum(axis=1), axis=0)              # each country's host mix, comparable across rows

cmap = plt.get_cmap("tab20")
fig, ax = plt.subplots(figsize=(10, 6))
left = pd.Series(0.0, index=frac.index)
for i, g in enumerate(frac.columns):
    ax.barh(frac.index, frac[g], left=left,
            color="#cccccc" if g == "Other" else cmap(i), label=g)
    left = left + frac[g]
for i, c in enumerate(frac.index):                 # n annotation at the right edge
    ax.text(1.005, i, f"n={ct.loc[c].sum():,}", va="center", fontsize=8)
ax.set_xlim(0, 1)
ax.set_xlabel("fraction of hosted plasmids")
ax.set_title("Host-genus composition by country (top 12 countries, non-simulated set)")
ax.legend(bbox_to_anchor=(1.13, 1), loc="upper left", fontsize=8, title="host genus")
plt.tight_layout()
plt.show()

ct[::-1]   # the underlying counts, biggest country first

In [ ]:
# Host species by habitat (hab_sub) — which bacterial host species carry plasmids in each REAL habitat.
# Restrict to plasmids that have a host species AND a real habitat (drop the 3 non-habitat "Unknown"
# buckets, which aren't habitats). host = the bacterium the plasmid lives in, not the sampled animal.
NON_HABITAT = {"Unlabelled", "Unresolved", "Geography only (no habitat)"}
hsub = real[real["host"].notna() & ~real["hab_sub"].isin(NON_HABITAT)]
print(f"{len(hsub):,} plasmids with a host species in a real habitat "
      f"(excluded non-habitat buckets: {sorted(NON_HABITAT)})")

top_habitats = hsub["hab_sub"].value_counts().head(12).index
top_species  = hsub["host"].value_counts().head(8).index

sub = hsub[hsub["hab_sub"].isin(top_habitats)].copy()
sub["sp"] = sub["host"].where(sub["host"].isin(top_species), "Other")

ct   = pd.crosstab(sub["hab_sub"], sub["sp"])[list(top_species) + ["Other"]]
ct   = ct.loc[top_habitats[::-1]]                  # biggest habitat last -> ends up at top in barh
frac = ct.div(ct.sum(axis=1), axis=0)              # each habitat's species mix, comparable across rows

cmap = plt.get_cmap("tab20")
fig, ax = plt.subplots(figsize=(11, 6))
left = pd.Series(0.0, index=frac.index)
for i, sp in enumerate(frac.columns):
    ax.barh(frac.index, frac[sp], left=left,
            color="#cccccc" if sp == "Other" else cmap(i), label=sp)
    left = left + frac[sp]
for i, h in enumerate(frac.index):                 # n annotation at the right edge
    ax.text(1.005, i, f"n={ct.loc[h].sum():,}", va="center", fontsize=8)
ax.set_xlim(0, 1)
ax.set_xlabel("fraction of hosted plasmids")
ax.set_title("Host-species composition by habitat (hab_sub, top 12 real habitats)")
ax.legend(bbox_to_anchor=(1.14, 1), loc="upper left", fontsize=8, title="host species")
plt.tight_layout()
plt.show()

ct[::-1]   # underlying counts, biggest habitat first

In [ ]:
# Two-tier One Health habitat labels on `real`:
#   oh_domain  = TOP tier   -> Human / Animal / Environment / Plant & food
#   oh_habitat = LOWER tier -> the raw hab_sub value, verbatim
# Both are NaN *together* when a plasmid's hab_sub carries no One Health domain (the non-habitat
# buckets Unlabelled / Unresolved / Geography only, and the 2 malformed micro-labels).
# The hab_sub -> domain map mirrors assign_domain() in scripts/analyze_amr_onehealth_mobility.py
# (the LOCKED taxonomy) — keep them in sync, don't diverge here.
HAB_DOMAIN = {
    "Human: unspecified": "Human", "Human: gut/faeces": "Human", "Human: blood": "Human",
    "Human: urine": "Human", "Human: respiratory": "Human", "Human: skin/wound": "Human",
    "Human: other clinical": "Human", "Human: oral": "Human",
    "Livestock: pig": "Animal", "Livestock: cattle": "Animal", "Livestock: other": "Animal",
    "Poultry/bird": "Animal", "Companion animal": "Animal", "Fish/aquaculture": "Animal",
    "Other mammal": "Animal", "Rodent": "Animal", "Insect/arthropod": "Animal",
    "Other invertebrate": "Animal", "Gut/faeces: unspecified host": "Animal",
    "Wastewater/sewage": "Environment", "Bioreactor": "Environment", "Built environment": "Environment",
    "Solid waste/compost": "Environment", "Industrial/remediation": "Environment",
    "Terrestrial/soil": "Environment", "Aquatic: freshwater/other": "Environment",
    "Aquatic: marine": "Environment", "Air": "Environment", "Extreme/other": "Environment",
    "Environmental: unspecified": "Environment",
    "Plant": "Plant & food", "Food/fermentation": "Plant & food", "Algae": "Plant & food",
    "Fungi": "Plant & food",
}
real["oh_domain"]  = real["hab_sub"].map(HAB_DOMAIN)
real["oh_habitat"] = real["hab_sub"].where(real["oh_domain"].notna())   # lower tier; NaN if no domain

n = real["oh_domain"].notna().sum()
print(f"One Health label set for {n:,} / {len(real):,} `real` plasmids "
      f"({real['oh_domain'].isna().sum():,} carry no One Health domain)\n")
print(real["oh_domain"].value_counts(dropna=False).to_string())

# the two-tier structure with per-habitat counts
(real.groupby(["oh_domain", "oh_habitat"], observed=True).size()
     .rename("n_plasmids").reset_index())

In [ ]:
# Plot the two-tier One Health structure: one bar per lower-tier habitat (oh_habitat),
# coloured by its top-tier domain (oh_domain), habitats grouped by domain and sorted within it.
from matplotlib.patches import Patch

DOMAIN_ORDER = ["Human", "Animal", "Environment", "Plant & food"]
DOMAIN_COLOR = {"Human": "#d1495b", "Animal": "#edae49",
                "Environment": "#00798c", "Plant & food": "#66a182"}

oh = real[real["oh_domain"].notna()]
counts = oh.groupby(["oh_domain", "oh_habitat"], observed=True).size().rename("n")

rows = []                                          # (domain, habitat, n), grouped by domain, n desc
for dom in DOMAIN_ORDER:
    for hab, n in counts.loc[dom].sort_values(ascending=False).items():
        rows.append((dom, hab, n))
rows = rows[::-1]                                  # barh draws bottom->top, so first domain ends on top

vals   = [n for _, _, n in rows]
colors = [DOMAIN_COLOR[d] for d, _, _ in rows]

fig, ax = plt.subplots(figsize=(9, 11))
ax.barh(range(len(rows)), vals, color=colors)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([hab for _, hab, _ in rows], fontsize=8)
for i, n in enumerate(vals):
    ax.text(n, i, f" {n:,}", va="center", fontsize=7)
ax.legend(handles=[Patch(color=DOMAIN_COLOR[d], label=d) for d in DOMAIN_ORDER],
          title="One Health domain", loc="lower right", fontsize=9)
ax.set_xlabel("plasmids")
ax.set_title(f"One Health habitats — {len(oh):,} labelled plasmids "
             f"({real['oh_domain'].isna().sum():,} unlabelled, not shown)")
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

In [ ]:
# Replicon prevalence from mob_typer calls (mob_rep_types). mob_typer ran on ~all plasmids; a replicon
# marker is reported only for those that carry a recognised one. Values are comma-separated replicon
# tokens and a plasmid can carry several, so we explode on ',' and count distinct replicons per plasmid.
from matplotlib.patches import Patch

rep = real["mob_rep_types"].dropna().str.split(",").explode().str.strip()
rep = rep[rep != ""]
carriers = rep.groupby(level=0).unique().explode()      # one entry per (plasmid, distinct replicon)
prev = carriers.value_counts()                           # plasmids carrying each replicon
prevalence = 100 * prev / len(real)                      # % of ALL real plasmids

n_typed = real["mob_rep_types"].notna().sum()
print(f"{len(real):,} real plasmids; {n_typed:,} ({n_typed/len(real):.1%}) carry >=1 mob replicon "
      f"marker; {prev.size:,} distinct replicon types")

top = prevalence.head(22)[::-1]                           # reverse so the most prevalent sits on top
is_novel = top.index.str.contains("rep_cluster")         # MOB novel/uncharacterised clusters
colors = ["#b0b0b0" if nv else "#7a3fbf" for nv in is_novel]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(len(top)), top.values, color=colors)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.index, fontsize=8)
for i, v in enumerate(top.values):
    ax.text(v, i, f" {v:.1f}%", va="center", fontsize=7)
ax.legend(handles=[Patch(color="#7a3fbf", label="named Inc/rep type"),
                   Patch(color="#b0b0b0", label="novel MOB cluster (rep_cluster_*)")],
          loc="lower right", fontsize=8)
ax.set_xlabel("prevalence (% of all real plasmids)")
ax.set_title(f"Replicon prevalence (mob_typer) — top {len(top)} of {prev.size:,} types")
ax.margins(x=0.13)
plt.tight_layout()
plt.show()

prev.head(22).rename("n_plasmids").to_frame().assign(pct=lambda d: (100*d.n_plasmids/len(real)).round(2))

In [ ]:
# Multireplicon plasmids = >=2 DISTINCT replicon types on one plasmid (mob_typer, mob_rep_types).
# Only plasmids with a replicon marker can be counted; the untyped 47% are neither single nor multi.
from matplotlib.patches import Patch

typed = real.loc[real["mob_rep_types"].notna(), "mob_rep_types"]
sets_ = typed.str.split(",").apply(lambda xs: sorted({x.strip() for x in xs if x.strip()}))
nrep  = sets_.apply(len)                                  # distinct replicon types per typed plasmid

n_typed = len(nrep)
n_multi = int((nrep >= 2).sum())
print(f"typed (>=1 replicon):    {n_typed:>7,}  ({n_typed/len(real):.1%} of {len(real):,} real)")
print(f"single-replicon (==1):   {(nrep==1).sum():>7,}  ({(nrep==1).mean():.1%} of typed)")
print(f"MULTIREPLICON (>=2):     {n_multi:>7,}  ({n_multi/n_typed:.1%} of typed, {n_multi/len(real):.1%} of real)")
print(f"max replicons on one plasmid: {nrep.max()}   median among multireplicon: {nrep[nrep>=2].median():.0f}")

# distribution: #plasmids by number of distinct replicon types (single grey vs multireplicon purple)
cap = 8
d = nrep.clip(upper=cap).value_counts().sort_index()
labels = [str(int(i)) for i in d.index]; labels[-1] = f"{cap}+"
colors = ["#b0b0b0" if i < 2 else "#7a3fbf" for i in d.index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(d)), d.values, color=colors)
ax.set_xticks(range(len(d))); ax.set_xticklabels(labels)
for i, v in enumerate(d.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("distinct replicon types on a plasmid (mob_typer)")
ax.set_ylabel("plasmids")
ax.set_title(f"Multireplicon carriage — {n_multi:,} plasmids carry >=2 replicons "
             f"({n_multi/n_typed:.0%} of {n_typed:,} typed)")
ax.legend(handles=[Patch(color="#b0b0b0", label="single (1)"),
                   Patch(color="#7a3fbf", label="multireplicon (>=2)")], fontsize=9)
plt.tight_layout()
plt.show()

# most common replicon COMBINATIONS among multireplicon plasmids
(sets_[nrep >= 2].apply(" + ".join).value_counts().head(15)
     .rename("n_plasmids").to_frame())

In [ ]:
# Replicon co-occurrence NETWORK (mob_typer). Nodes = replicons; an edge means the two co-occur on the
# same plasmid MORE than expected by chance. For every pair of candidate replicons we run a one-sided
# Fisher exact test (alternative="greater") on the 2x2 over typed plasmids, BH-correct across all pairs,
# and keep an edge only if it is significant, enriched, and well supported.
import numpy as np
from itertools import combinations
from collections import Counter
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import networkx as nx

MIN_OCC, MIN_CO, OR_MIN, Q_MAX = 50, 20, 2.0, 0.05   # tune these to make the graph denser/sparser

typed = real.loc[real["mob_rep_types"].notna(), "mob_rep_types"]
sets_ = typed.str.split(",").apply(lambda xs: sorted({x.strip() for x in xs if x.strip()}))
N = len(sets_)
occ = Counter(r for s in sets_ for r in s)
cand = sorted([r for r, c in occ.items() if c >= MIN_OCC])           # tested replicons
idx  = {r: i for i, r in enumerate(cand)}

P = np.zeros((N, len(cand)), dtype=np.int32)                          # presence matrix
for row, s in enumerate(sets_):
    for r in s:
        if r in idx:
            P[row, idx[r]] = 1
co = P.T @ P                                                          # co[i,j] = #plasmids with both
occ_arr = np.diag(co)

pairs, pvals, ors, boths = [], [], [], []
for i, j in combinations(range(len(cand)), 2):
    both = co[i, j]
    if both == 0:                                                    # can't be positively enriched
        continue
    a, b = occ_arr[i] - both, occ_arr[j] - both
    orr, p = fisher_exact([[both, a], [b, N - both - a - b]], alternative="greater")
    pairs.append((cand[i], cand[j])); pvals.append(p); ors.append(orr); boths.append(int(both))

q = multipletests(pvals, method="fdr_bh")[1]
edges = [(r1, r2, orr, bo, qq) for (r1, r2), orr, bo, qq in zip(pairs, ors, boths, q)
         if qq < Q_MAX and orr >= OR_MIN and bo >= MIN_CO]
print(f"N typed plasmids={N:,}; candidate replicons (>= {MIN_OCC} occ)={len(cand)}; "
      f"pairs tested={len(pairs):,}; significant co-occurrence edges="
      f"(q<{Q_MAX}, OR>={OR_MIN}, both>={MIN_CO}) -> {len(edges)}")

# perfect co-occurrence gives OR=inf; bound it so log2 stays finite for layout + edge widths
def logw(orr):
    return min(np.log2(orr), 12.0) if np.isfinite(orr) else 12.0
G = nx.Graph()
for r1, r2, orr, bo, qq in edges:
    G.add_edge(r1, r2, orr=orr, both=bo, w=logw(orr))
print(f"graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
      f"{nx.number_connected_components(G)} modules")

pos = nx.spring_layout(G, k=0.6, seed=1, weight="w")
node_col = ["#b0b0b0" if "rep_cluster" in n else "#7a3fbf" for n in G.nodes]
node_sz  = [40 + 4*np.sqrt(occ[n]) for n in G.nodes]
ew = np.array([G[u][v]["w"] for u, v in G.edges])

fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_edges(G, pos, width=0.4 + 2.5*ew/ew.max(), edge_color="#9aa0a6", alpha=0.6, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_col, node_size=node_sz, linewidths=0.4,
                       edgecolors="white", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#7a3fbf", label="named Inc/rep type"),
                   Patch(color="#b0b0b0", label="novel MOB cluster")],
          loc="lower left", fontsize=9)
ax.set_title(f"Statistically significant replicon co-occurrence (mob_typer)\n"
             f"Fisher exact, BH q<{Q_MAX}, OR>={OR_MIN}, >={MIN_CO} plasmids share the pair "
             f"— node size ~ frequency, edge width ~ log2(OR)")
ax.axis("off")
plt.tight_layout()
plt.show()

# strongest associations
(pd.DataFrame(edges, columns=["replicon_A", "replicon_B", "odds_ratio", "n_both", "q"])
   .sort_values("odds_ratio", ascending=False).head(20).reset_index(drop=True))

In [ ]:
# Is multireplicon carriage more prevalent in some hosts / One Health categories?
# Rate = fraction of TYPED plasmids (>=1 mob replicon marker) that are multireplicon (>=2). Conditioning
# on "typed" removes replicon-typing-coverage differences, so contrasts reflect biology. Each group is
# Fisher-tested vs all other groups in the same stratification; BH-corrected; Wilson 95% CIs.
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

t = real[real["mob_rep_types"].notna()].copy()
t["is_multi"] = t["mob_rep_types"].str.split(",").apply(
    lambda xs: len({x.strip() for x in xs if x.strip()}) >= 2)
base = t["is_multi"].mean()

def enrich(df, col, min_n):
    tot_n, tot_k = len(df), int(df["is_multi"].sum())
    rows = []
    for g, s in df.groupby(col, observed=True):
        n, k = len(s), int(s["is_multi"].sum())
        if n < min_n:
            continue
        orr, p = fisher_exact([[k, n - k], [tot_k - k, (tot_n - n) - (tot_k - k)]])
        lo, hi = proportion_confint(k, n, method="wilson")
        rows.append({col: g, "n_typed": n, "n_multi": k, "rate": k / n,
                     "ci_lo": lo, "ci_hi": hi, "OR_vs_rest": orr, "p": p})
    out = pd.DataFrame(rows)
    out["q"] = multipletests(out["p"], method="fdr_bh")[1]
    return out.sort_values("rate", ascending=False).reset_index(drop=True)

host_tab = enrich(t[t["host_genus"].notna()], "host_genus", 100)
dom_tab  = enrich(t[t["oh_domain"].notna()],  "oh_domain",  30)
hab_tab  = enrich(t[t["oh_habitat"].notna()], "oh_habitat", 50)

def forest(ax, tab, col, title):
    d = tab.sort_values("rate")
    y = np.arange(len(d))
    # clip at 0: for rate==0 groups the Wilson lower bound can round a hair above 0, giving a tiny
    # negative rate-ci_lo that matplotlib rejects as xerr.
    xerr = np.clip(np.vstack([d.rate - d.ci_lo, d.ci_hi - d.rate]), 0, None)
    colors = np.where((d.q < 0.05) & (d.rate > base), "#d1495b",
             np.where((d.q < 0.05) & (d.rate < base), "#2a78d6", "#b0b0b0"))
    ax.errorbar(d.rate, y, xerr=xerr, fmt="none", ecolor="#999", elinewidth=1, capsize=2, zorder=1)
    ax.scatter(d.rate, y, c=colors, s=42, zorder=2, edgecolor="white", linewidth=0.5)
    ax.axvline(base, color="k", ls="--", lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{g}  (n={n:,})" for g, n in zip(d[col], d.n_typed)], fontsize=8)
    ax.set_xlabel("multireplicon rate")
    ax.set_title(title, fontsize=10)
    ax.margins(y=0.02)

fig, axes = plt.subplots(1, 2, figsize=(14, 8), gridspec_kw={"width_ratios": [1.3, 1]})
forest(axes[0], host_tab, "host_genus", f"Multireplicon rate by host genus (n>=100)")
forest(axes[1], dom_tab,  "oh_domain",  "by One Health domain")
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="#d1495b", label="elevated (q<0.05)"),
                        Patch(color="#2a78d6", label="depleted (q<0.05)"),
                        Patch(color="#b0b0b0", label="n.s.")],
               loc="lower right", fontsize=8, title=f"vs baseline {base:.1%}")
fig.suptitle(f"Multireplicon carriage vs baseline {base:.1%} (dashed) — typed plasmids only", y=1.01)
plt.tight_layout()
plt.show()

print("=== One Health domain ===");  print(dom_tab.round(3).to_string(index=False))
print("\n=== One Health habitat (oh_habitat) ==="); print(hab_tab.round(3).to_string(index=False))
host_tab.round(3)   # full host-genus table (sorted high -> low rate)

In [ ]:
# Are multireplicon plasmids more likely to carry AMR genes? amr_pos = CARD (Perfect+Strict) ARG count>0.
# The raw association is confounded by plasmid SIZE (multireplicon plasmids are bigger, and bigger
# plasmids carry more genes), so we also fit a size+GC-adjusted logistic for the INDEPENDENT effect.
import statsmodels.formula.api as smf
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportion_confint

t = real[real["mob_rep_types"].notna()].copy()
t["nrep"]     = t["mob_rep_types"].str.split(",").apply(lambda xs: len({x.strip() for x in xs if x.strip()}))
t["is_multi"] = (t["nrep"] >= 2).astype(int)
t["amr_pos"]  = (t["card_n_arg"] > 0).astype(int)

k1, n1 = int(t.loc[t.is_multi == 1, "amr_pos"].sum()), int((t.is_multi == 1).sum())
k0, n0 = int(t.loc[t.is_multi == 0, "amr_pos"].sum()), int((t.is_multi == 0).sum())
orr, p = fisher_exact([[k1, n1 - k1], [k0, n0 - k0]])
print(f"AMR+ | multireplicon (>=2):  {k1:,}/{n1:,} = {k1/n1:.1%}")
print(f"AMR+ | single-replicon (==1): {k0:,}/{n0:,} = {k0/n0:.1%}")
print(f"raw odds ratio = {orr:.2f}  (Fisher p={p:.1e})")

t["log10_size"] = np.log10(t["size_bp"].clip(lower=1))
fit = smf.logit("amr_pos ~ is_multi + log10_size + gc_percent",
                data=t.dropna(subset=["gc_percent"])).fit(disp=0)
aor = np.exp(fit.params["is_multi"]); lo, hi = np.exp(fit.conf_int().loc["is_multi"])
print(f"size+GC-adjusted logistic: multireplicon OR = {aor:.2f} (95% CI {lo:.2f}-{hi:.2f}); "
      f"size OR per log10(bp) = {np.exp(fit.params['log10_size']):.2f}")

# dose-response: AMR+ rate by number of distinct replicons (Wilson 95% CIs)
cap = 6
g = t.assign(nrep_c=t["nrep"].clip(upper=cap))
rows = []
for r, s in g.groupby("nrep_c"):
    k, n = int(s.amr_pos.sum()), len(s)
    lo2, hi2 = proportion_confint(k, n, method="wilson")
    rows.append((int(r), n, k / n, lo2, hi2))
dr = pd.DataFrame(rows, columns=["nrep", "n", "rate", "lo", "hi"])
labels = [str(r) for r in dr.nrep]; labels[-1] = f"{cap}+"

fig, ax = plt.subplots(figsize=(8, 5))
err = np.clip(np.vstack([dr.rate - dr.lo, dr.hi - dr.rate]), 0, None)
ax.bar(range(len(dr)), dr.rate, yerr=err, capsize=3,
       color=["#b0b0b0"] + ["#7a3fbf"] * (len(dr) - 1))
ax.axhline(t.amr_pos.mean(), color="k", ls="--", lw=1, label=f"overall {t.amr_pos.mean():.0%}")
for i, (r, n) in enumerate(zip(dr.rate, dr.n)):
    ax.text(i, r, f"{r:.0%}\n(n={n:,})", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(dr))); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.08)
ax.set_xlabel("distinct replicon types on plasmid (mob_typer)")
ax.set_ylabel("AMR+ rate (CARD Perfect+Strict)")
ax.set_title(f"AMR carriage rises with replicon count — raw OR {orr:.1f}, size-adjusted OR {aor:.1f}")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

dr.assign(rate=dr.rate.round(3))   # dose-response table

In [ ]:
# ANALYSIS 1 — replicon-type prevalence across habitats (hab_sub). Replicons aren't mutually exclusive
# (a plasmid can carry several), so we show P(carry replicon | typed plasmid in group) as a heatmap, NOT
# a stacked bar. `replicon_by` + `heat` are reused by the country cell below.
def replicon_by(cat, top_cat=12, top_rep=16, min_group=50, drop=None):
    d = real[real["mob_rep_types"].notna() & real[cat].notna()].copy()
    if drop:
        d = d[~d[cat].isin(drop)]
    d["reps"] = d["mob_rep_types"].str.split(",").apply(lambda xs: sorted({x.strip() for x in xs if x.strip()}))
    gsize = d.groupby(cat).size()
    cats = gsize[gsize >= min_group].sort_values(ascending=False).head(top_cat).index
    long = d[d[cat].isin(cats)][[cat, "reps"]].explode("reps")
    top = long["reps"].value_counts().head(top_rep).index                 # commonest replicons in scope
    counts = long[long["reps"].isin(top)].groupby([cat, "reps"]).size().unstack(fill_value=0)
    prev = counts.div(gsize.loc[counts.index], axis=0) * 100              # % of group's typed plasmids
    return prev.loc[cats][top], gsize.loc[cats]

def heat(prev, gsize, title):
    fig, ax = plt.subplots(figsize=(0.62 * prev.shape[1] + 4, 0.55 * prev.shape[0] + 2))
    im = ax.imshow(prev.values, aspect="auto", cmap="viridis")
    thr = prev.values.max() * 0.55
    for i in range(prev.shape[0]):
        for j in range(prev.shape[1]):
            v = prev.values[i, j]
            if v >= 1:
                ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6,
                        color="white" if v < thr else "black")
    ax.set_xticks(range(prev.shape[1])); ax.set_xticklabels(prev.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(prev.shape[0]))
    ax.set_yticklabels([f"{c}  (n={gsize[c]:,})" for c in prev.index], fontsize=8)
    fig.colorbar(im, ax=ax, label="% of typed plasmids carrying replicon", shrink=0.6)
    ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()

NON_HABITAT = {"Unlabelled", "Unresolved", "Geography only (no habitat)"}
prev_hab, g_hab = replicon_by("hab_sub", drop=NON_HABITAT)
heat(prev_hab, g_hab, "Replicon-type prevalence by habitat (top 12 hab_sub x top 16 replicons)")
prev_hab.round(1)

In [ ]:
# ANALYSIS 2 — replicon-type prevalence across countries (geo_country). Same P(carry replicon | typed
# plasmid in country) heatmap; reuses replicon_by / heat from the cell above (run that one first).
prev_ctry, g_ctry = replicon_by("geo_country")
heat(prev_ctry, g_ctry, "Replicon-type prevalence by country (top 12 countries x top 16 replicons)")
prev_ctry.round(1)

In [ ]:
# Multireplicon prevalence by COUNTRY and by HABITAT (oh_habitat). Reuses enrich() / forest() / base / t
# from the host+domain cell above (run that one first). CAVEAT: country rate is heavily confounded by
# WHAT each country contributed (clinical Enterobacterales surveillance vs environmental/agri sampling),
# so read it as sampling composition, not a geographic biology signal.
country_tab = enrich(t[t["geo_country"].notna()], "geo_country", 100)
habf_tab    = enrich(t[t["oh_habitat"].notna()],  "oh_habitat",  50)

sig_leg = [Patch(color="#d1495b", label="elevated (q<0.05)"),
           Patch(color="#2a78d6", label="depleted (q<0.05)"),
           Patch(color="#b0b0b0", label="n.s.")]

fig, ax = plt.subplots(figsize=(7.5, 0.30 * len(country_tab) + 1.5))
forest(ax, country_tab, "geo_country", f"Multireplicon rate by country (n>=100) — baseline {base:.1%}")
ax.legend(handles=sig_leg, loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7.5, 0.30 * len(habf_tab) + 1.5))
forest(ax, habf_tab, "oh_habitat", f"Multireplicon rate by habitat (oh_habitat, n>=50) — baseline {base:.1%}")
ax.legend(handles=sig_leg, loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

print("=== country (sorted high -> low) ==="); print(country_tab.round(3).to_string(index=False))
habf_tab.round(3)

In [ ]:
# Host range predicted by mob_typer. master kept only the taxon NAME (mob_host_range); the RANK — the
# breadth ladder genus<family<order<class<phylum<multi-phylla — lives in mob_full. Join it on (rank "-"
# = no prediction -> NaN) and store hr_rank on `real` as an ordered categorical.
mob_full = data["mob_full"]
BREADTH = ["genus", "family", "order", "class", "phylum", "multi-phylla"]
rank_map = dict(zip(mob_full["sample_id"], mob_full["predicted_host_range_overall_rank"]))
hr = real["plasmid_id"].map(rank_map).replace("-", np.nan)
real["hr_rank"] = pd.Categorical(hr, categories=BREADTH, ordered=True)

n_pred = int(real["hr_rank"].notna().sum())
print(f"host-range prediction for {n_pred:,}/{len(real):,} ({n_pred/len(real):.1%}) real plasmids "
      f"(mob_typer returns '-' for the rest)")

dist = real["hr_rank"].value_counts().reindex(BREADTH)
cmap = plt.get_cmap("RdYlBu_r")
colors = [cmap(i / (len(BREADTH) - 1)) for i in range(len(BREADTH))]   # narrow=blue -> broad=red

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.1]})
# A — breadth distribution (narrow -> broad)
ax1.bar(range(len(dist)), dist.values, color=colors)
ax1.set_xticks(range(len(dist))); ax1.set_xticklabels(BREADTH, rotation=30, ha="right")
for i, v in enumerate(dist.values):
    ax1.text(i, v, f"{v:,}\n{v/n_pred:.0%}", ha="center", va="bottom", fontsize=8)
ax1.set_ylabel("plasmids"); ax1.set_xlabel("predicted host-range breadth (narrow -> broad)")
ax1.set_title(f"Host-range breadth (mob_typer) — {n_pred:,} predicted")
ax1.margins(y=0.16)

# B — breadth composition within each mobility class (row-normalised)
ct = pd.crosstab(real["predicted_mobility"], real["hr_rank"]).reindex(columns=BREADTH).fillna(0)
ct = ct.reindex(["conjugative", "mobilizable", "non-mobilizable"])
frac = ct.div(ct.sum(axis=1), axis=0)
left = np.zeros(len(ct))
for i, b in enumerate(BREADTH):
    ax2.barh(range(len(ct)), frac[b].values, left=left, color=colors[i], label=b)
    left = left + frac[b].values
ax2.set_yticks(range(len(ct)))
ax2.set_yticklabels([f"{m}\n(n={int(ct.loc[m].sum()):,})" for m in ct.index], fontsize=8)
ax2.set_xlim(0, 1); ax2.set_xlabel("fraction of plasmids")
ax2.set_title("Host-range breadth by mobility class")
ax2.legend(fontsize=7, ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.12), title="breadth")
plt.tight_layout(); plt.show()

# the actual predicted host-range taxa (names)
real["mob_host_range"].value_counts().head(15).rename("n_plasmids").to_frame()

In [ ]:
# ASSOCIATION: replicon type vs predicted host-range breadth. breadth = ordinal 1..6 (genus..multi-phylla)
# from hr_rank. For each replicon we take the mean breadth of the plasmids carrying it -> which replicons
# ride on BROAD vs NARROW host-range backbones. Needs the host-range cell above (hr_rank).
from matplotlib.patches import Patch
BREADTH = ["genus", "family", "order", "class", "phylum", "multi-phylla"]
h = real[real["hr_rank"].notna() & real["mob_rep_types"].notna()].copy()
h["breadth"] = h["hr_rank"].cat.codes + 1                 # 1..6
mean_all = h["breadth"].mean()

long = (h.assign(reps=h["mob_rep_types"].str.split(",").apply(lambda xs: sorted({x.strip() for x in xs if x.strip()})))
         [["reps", "breadth"]].explode("reps"))
g = long.groupby("reps")["breadth"].agg(n="size", mean="mean")
g = g[g["n"] >= 100].sort_values("mean")
show = pd.concat([g.head(15), g.tail(15)])               # 15 narrowest + 15 broadest replicons

cols = ["#7a3fbf" if "rep_cluster" not in r else "#b0b0b0" for r in show.index]
fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(range(len(show)), show["mean"], color=cols)
ax.axvline(mean_all, color="k", ls="--", lw=1)
ax.set_yticks(range(len(show)))
ax.set_yticklabels([f"{r}  (n={int(n):,})" for r, n in zip(show.index, show["n"])], fontsize=7)
ax.set_xlim(1, 6.2); ax.set_xticks(range(1, 7)); ax.set_xticklabels(BREADTH, rotation=30, ha="right", fontsize=8)
ax.set_xlabel("mean predicted host-range breadth")
ax.set_title(f"Replicon type vs host-range breadth — 15 narrowest + 15 broadest (n>=100)\n"
             f"dashed = overall mean {mean_all:.2f}")
ax.legend(handles=[Patch(color="#7a3fbf", label="named Inc/rep type"),
                   Patch(color="#b0b0b0", label="novel MOB cluster")], loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

g.sort_values("mean", ascending=False).round(2)          # full ranked table (broad -> narrow)

In [ ]:
# Do MULTIREPLICON plasmids have a BIGGER host range? Reuses h / breadth / mean_all / BREADTH from the
# cell above (run it first). breadth ordinal 1..6; test with Mann-Whitney (ordinal, non-parametric).
from scipy.stats import mannwhitneyu

h["nrep"] = h["mob_rep_types"].str.split(",").apply(lambda xs: len({x.strip() for x in xs if x.strip()}))
h["is_multi"] = h["nrep"] >= 2
s, mm = h.loc[~h.is_multi, "breadth"], h.loc[h.is_multi, "breadth"]
u, p = mannwhitneyu(mm, s, alternative="two-sided")
print(f"single-replicon:  mean breadth {s.mean():.2f} (median {s.median():.0f}), n={len(s):,}")
print(f"multireplicon:    mean breadth {mm.mean():.2f} (median {mm.median():.0f}), n={len(mm):,}")
print(f"Mann-Whitney U two-sided p = {p:.1e}  -> multireplicon host range is "
      f"{'BROADER' if mm.mean() > s.mean() else 'narrower'} on average")

cmap = plt.get_cmap("RdYlBu_r"); bcol = [cmap(i / 5) for i in range(6)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.15]})

# A — dose-response: mean breadth by replicon count (95% CI = 1.96*SEM)
cap = 5
gg = h.assign(nc=h["nrep"].clip(upper=cap)).groupby("nc")["breadth"].agg(["mean", "sem", "size"])
labels = [str(int(i)) for i in gg.index]; labels[-1] = f"{cap}+"
ax1.bar(range(len(gg)), gg["mean"], yerr=1.96 * gg["sem"], capsize=3, color="#7a3fbf")
ax1.axhline(mean_all, color="k", ls="--", lw=1, label=f"overall {mean_all:.2f}")
for i, (mv, nv) in enumerate(zip(gg["mean"], gg["size"])):
    ax1.text(i, mv, f"{mv:.2f}\n(n={int(nv):,})", ha="center", va="bottom", fontsize=8)
ax1.set_xticks(range(len(gg))); ax1.set_xticklabels(labels)
ax1.set_ylim(1, 6); ax1.set_xlabel("distinct replicon types on plasmid")
ax1.set_ylabel("mean host-range breadth (1=genus .. 6=multi-phylla)")
ax1.set_title("Host-range breadth vs replicon count"); ax1.legend(fontsize=8)

# B — breadth composition, single vs multireplicon (stacked, row-normalised)
ct = pd.crosstab(h["is_multi"].map({False: "single", True: "multireplicon"}), h["hr_rank"])
ct = ct.reindex(columns=BREADTH).reindex(["single", "multireplicon"])
frac = ct.div(ct.sum(axis=1), axis=0)
left = np.zeros(len(ct))
for i, b in enumerate(BREADTH):
    ax2.barh(range(len(ct)), frac[b].values, left=left, color=bcol[i], label=b)
    left = left + frac[b].values
ax2.set_yticks(range(len(ct)))
ax2.set_yticklabels([f"{m}\n(n={int(ct.loc[m].sum()):,})" for m in ct.index], fontsize=8)
ax2.set_xlim(0, 1); ax2.set_xlabel("fraction of plasmids")
ax2.set_title("Breadth composition: single vs multireplicon")
ax2.legend(fontsize=7, ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.12), title="breadth")
plt.tight_layout(); plt.show()

gg.round(3)

In [ ]:
# SPREAD OF ARGs ACROSS ENVIRONMENTS (One Health domains). Which resistance genes are shared across
# Human / Animal / Environment / Plant&food vs confined to one? Uses CARD ARO terms (card_aro_list) on
# AMR+ plasmids that carry a domain. CAVEAT: domains are unevenly sampled (Human >> others), so "seen in
# a domain" is easier for Human; the all-4-domain ARGs are robust (they surface despite small plant/env n).
DOMS = ["Human", "Animal", "Environment", "Plant & food"]
d = real[real["card_aro_list"].notna() & real["oh_domain"].notna()].copy()
d["args"] = d["card_aro_list"].str.split(";").apply(lambda xs: sorted({x.strip() for x in xs if x.strip()}))
long = d[["oh_domain", "args"]].explode("args")
print(f"{len(d):,} AMR+ plasmids with a domain; {long['args'].nunique():,} distinct ARGs")
print("AMR+ by domain:", dict(d["oh_domain"].value_counts().reindex(DOMS)))

n_dom  = long.groupby("args")["oh_domain"].nunique()      # how many domains each ARG reaches (1..4)
n_plas = long.groupby("args").size()
repert = {dm: set(long.loc[long.oh_domain == dm, "args"]) for dm in DOMS}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.15]})
# A — spread breadth: #ARGs by number of domains they appear in
vc = n_dom.value_counts().sort_index()
cmap = plt.get_cmap("YlOrRd")
ax1.bar(vc.index, vc.values, color=[cmap(0.25 + 0.6 * (i - 1) / 3) for i in vc.index])
for x, v in zip(vc.index, vc.values):
    ax1.text(x, v, f"{v}\n{v/n_dom.size:.0%}", ha="center", va="bottom", fontsize=9)
ax1.set_xticks(range(1, 5)); ax1.set_xlabel("number of One Health domains an ARG appears in")
ax1.set_ylabel("distinct ARGs"); ax1.margins(y=0.16)
ax1.set_title(f"ARG environmental spread — {int((n_dom == 4).sum())} of {n_dom.size} ARGs span ALL 4 domains")

# B — ARG-repertoire Jaccard between domains (diagonal shows each domain's ARG richness)
J = np.full((4, 4), np.nan)
for i, a in enumerate(DOMS):
    for j, b in enumerate(DOMS):
        if i != j:
            J[i, j] = len(repert[a] & repert[b]) / len(repert[a] | repert[b])
im = ax2.imshow(J, cmap="viridis")
thr = np.nanmax(J) * 0.6
for i in range(4):
    for j in range(4):
        if i == j:
            ax2.text(j, i, f"{len(repert[DOMS[i]])}\nARGs", ha="center", va="center", fontsize=7, color="dimgray")
        else:
            ax2.text(j, i, f"{J[i, j]:.2f}", ha="center", va="center", fontsize=9,
                     color="white" if J[i, j] < thr else "black")
ax2.set_xticks(range(4)); ax2.set_xticklabels(DOMS, fontsize=8, rotation=20, ha="right")
ax2.set_yticks(range(4)); ax2.set_yticklabels(DOMS, fontsize=8)
fig.colorbar(im, ax=ax2, label="Jaccard (shared fraction of ARG repertoires)", shrink=0.7)
ax2.set_title("ARG-repertoire overlap between domains")
plt.tight_layout(); plt.show()

# most widely-disseminated ARGs (present in all 4 domains), by #plasmids carrying them
(n_plas[n_dom == 4].sort_values(ascending=False).head(20).rename("n_plasmids").to_frame())